In [ ]:
# =====================================================================
# BOLUM 0 - Ayarlar, veri ve DONDURULMUS bolmenin yuklenmesi
# ONEMLI: Bu kod yeni bir train/test ayrimi URETMEZ.
# =====================================================================
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

RANDOM_STATE = 42
N_SPLITS = 5
ROUND_DEC = 6
EPS = 1e-8                 # sifira bolunmeye karsi guvenlik esigi

# ---- Portable project paths ---------------------------------------------
from pathlib import Path

def find_project_root():
    """Locate the repository root from the current working directory."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "masonry_tower_primary_dataset.xlsx").is_file():
            return candidate
    raise FileNotFoundError(
        "Project root could not be located. Run this notebook from the repository "
        "root or from its code/ directory, and keep the data/ directory unchanged."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAT_FILE = DATA_DIR / "masonry_tower_primary_dataset.xlsx"
# ------------------------------------------------------------------------
SPLIT_FILE = os.path.join(OUTPUT_DIR, "Data_Split_Assignment.xlsx")

COMP_DIR = os.path.join(OUTPUT_DIR, "Model_Comparison")
AVP_DIR  = os.path.join(OUTPUT_DIR, "Actual_vs_Predicted")
RES_DIR  = os.path.join(OUTPUT_DIR, "Residual_Plots")
for k in [OUTPUT_DIR, COMP_DIR, AVP_DIR, RES_DIR]:
    os.makedirs(k, exist_ok=True)

# ---- Degisken tanimlari (Asama 3 ile ayni sira) -----------------------
GEO_COLS = ["Height (m)", "Section a (m)", "Section b (m)", "Wall Thickness (m)",
            "Opening z/H", "Opening Ratio x (%)", "Opening Ratio y (%)"]
MAT_COLS = ["E (MPa)", "d (kg/m3)"]
FEATURE_COLS = GEO_COLS + MAT_COLS
TARGETS = ["f1 (Hz)", "f2 (Hz)"]
TARGET_KISA = {"f1 (Hz)": "f1", "f2 (Hz)": "f2"}

BEKLENEN_DISARIDA = {"G170", "G187", "G183", "G068", "G099", "G076"}

# ---- Grafik ayarlari --------------------------------------------------
DPI = 300
sns.set_style("white")
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.titlesize": 12, "axes.labelsize": 12,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 9.5,
    "axes.linewidth": 0.9, "savefig.dpi": DPI, "savefig.bbox": "tight",
})
RENK_CV, RENK_TEST, RENK_VURGU = "#4C72B0", "#55A868", "#C44E52"


def kaydet(fig, klasor, ad):
    """Figuru PNG olarak kaydeder ve bellekten temizler."""
    yol = os.path.join(klasor, ad + ".png")
    fig.savefig(yol, dpi=DPI)
    plt.close(fig)
    print("Kaydedildi:", yol)


def geometry_id_olustur(veri, geo_cols=GEO_COLS, ndec=ROUND_DEC):
    """Asama 1 ve 3 ile birebir ayni Geometry_ID uretimi."""
    anahtar = veri[geo_cols].round(ndec).astype(str).agg("|".join, axis=1)
    esleme = {k: f"G{i+1:03d}" for i, k in enumerate(pd.unique(anahtar))}
    return anahtar.map(esleme)


def dondurulmus_bolmeyi_yukle(veri, split_file=SPLIT_FILE):
    """Onceden kaydedilmis Split ve CV_Fold atamalarini okur ve dogrular."""
    if not os.path.exists(split_file):
        raise FileNotFoundError(
            f"Bolme dosyasi bulunamadi: {split_file}\n"
            "Asama 3 kodunu calistirip Data_Split_Assignment.xlsx dosyasini uretin. "
            "Bu kod yeni bir bolme URETMEZ.")

    atama = pd.read_excel(split_file, sheet_name="Row_assignment")
    if len(atama) != len(veri):
        raise ValueError(f"Satir sayisi uyusmuyor: veri={len(veri)}, bolme={len(atama)}")

    atama = atama.sort_values("Row_index").reset_index(drop=True)
    if not (atama["Geometry_ID"].values == veri["Geometry_ID"].values).all():
        raise ValueError("Geometry_ID sirasi bolme dosyasiyla uyusmuyor. "
                         "Ayni Excel dosyasi ve ayni satir sirasi kullanilmalidir.")

    veri = veri.copy()
    veri["Split"] = atama["Split"].values
    veri["CV_Fold"] = atama["CV_Fold"].values

    # Beklenen yapinin dogrulanmasi
    n_tr = int((veri["Split"] == "Train").sum())
    n_te = int((veri["Split"] == "Test").sum())
    g_tr = veri.loc[veri["Split"] == "Train", "Geometry_ID"].nunique()
    g_te = veri.loc[veri["Split"] == "Test", "Geometry_ID"].nunique()
    beklenen = [("Train records", 912, n_tr), ("Test records", 234, n_te),
                ("Train geometries", 152, g_tr), ("Test geometries", 39, g_te)]
    for ad, bek, goz in beklenen:
        if bek != goz:
            raise ValueError(f"Bolme yapisi beklenenden farkli -> {ad}: beklenen {bek}, gozlenen {goz}")

    # Sizinti kontrolu
    ortak = (set(veri.loc[veri["Split"] == "Train", "Geometry_ID"]) &
             set(veri.loc[veri["Split"] == "Test", "Geometry_ID"]))
    if ortak:
        raise ValueError(f"VERI SIZINTISI: {len(ortak)} geometri hem egitimde hem testte.")

    print("Dondurulmus bolme yuklendi ve dogrulandi (912/234 kayit, 152/39 geometri).")
    return veri


# ---- Veri yukleme -----------------------------------------------------
df = pd.read_excel(MAT_FILE, sheet_name=0)
df["Geometry_ID"] = geometry_id_olustur(df)
df = dondurulmus_bolmeyi_yukle(df)

train_mask = (df["Split"] == "Train").values
test_mask = (df["Split"] == "Test").values

X_train = df.loc[train_mask, FEATURE_COLS].reset_index(drop=True)
X_test = df.loc[test_mask, FEATURE_COLS].reset_index(drop=True)
fold_train = df.loc[train_mask, "CV_Fold"].astype(int).reset_index(drop=True)
geo_train = df.loc[train_mask, "Geometry_ID"].reset_index(drop=True)
geo_test = df.loc[test_mask, "Geometry_ID"].reset_index(drop=True)
y_train = {t: df.loc[train_mask, t].reset_index(drop=True) for t in TARGETS}
y_test = {t: df.loc[test_mask, t].reset_index(drop=True) for t in TARGETS}

# Guvenlik: Geometry_ID ozellik olarak kullanilmamalidir
if "Geometry_ID" in X_train.columns:
    raise ValueError("HATA: Geometry_ID ozellik matrisinde bulunuyor.")

# ---- Tasarim uzayi disindaki test kayitlarinin belirlenmesi -----------
egitim_araligi = pd.DataFrame({"Feature": FEATURE_COLS,
                               "Train_min": X_train.min().values,
                               "Train_max": X_train.max().values})

disarida_mask = np.zeros(len(X_test), dtype=bool)
for kol in FEATURE_COLS:
    disarida_mask |= (X_test[kol] < X_train[kol].min()).values
    disarida_mask |= (X_test[kol] > X_train[kol].max()).values

outside_flag = np.where(disarida_mask, "Yes", "No")
bulunan = set(geo_test[disarida_mask].unique())
print(f"\nEgitim araligi disindaki test kayitlari: {int(disarida_mask.sum())} / {len(X_test)}")
print("Etkilenen geometriler:", sorted(bulunan))
if bulunan != BEKLENEN_DISARIDA:
    print("UYARI: Otomatik belirlenen geometri kumesi Asama 3'te bildirilen kumeden farkli!")
    print("  Beklenen:", sorted(BEKLENEN_DISARIDA))

In [ ]:
# =====================================================================
# BOLUM 1 - Temel modeller ve Pipeline fabrikasi
# =====================================================================

# Model adi -> (kisa ad, olcekleme gerekli mi, uretici fonksiyon)
def model_tanimlari():
    """Her cagrildiginda TAZE model nesneleri dondurur (fold'lar arasi kirlenmeyi onler)."""
    return {
        "Linear Regression":            ("LR",    True,  lambda: LinearRegression()),
        "Ridge Regression":             ("Ridge", True,  lambda: Ridge(alpha=1.0,solver='lsqr')),
        "Support Vector Regression":    ("SVR",   True,  lambda: SVR(kernel="rbf", C=1.0,
                                                                    epsilon=0.1, gamma="scale")),
        "K-Nearest Neighbors":          ("KNN",   True,  lambda: KNeighborsRegressor(
                                                            n_neighbors=5, weights="uniform",
                                                            metric="minkowski", p=2)),
        "Decision Tree Regression":     ("DT",    False, lambda: DecisionTreeRegressor(
                                                            random_state=RANDOM_STATE)),
        "Random Forest Regression":     ("RF",    False, lambda: RandomForestRegressor(
                                                            n_estimators=200, n_jobs=-1,
                                                            random_state=RANDOM_STATE)),
        "Gradient Boosting Regression": ("GBR",   False, lambda: GradientBoostingRegressor(
                                                            random_state=RANDOM_STATE)),
    }


def pipeline_olustur(model_adi):
    """Olcekleme gerekiyorsa StandardScaler -> Model, gerekmiyorsa yalnizca Model."""
    _, olcekle, uretici = model_tanimlari()[model_adi]
    adimlar = [("scaler", StandardScaler())] if olcekle else []
    adimlar.append(("model", uretici()))
    return Pipeline(adimlar)


MODELLER = list(model_tanimlari().keys())
KISA_AD = {ad: model_tanimlari()[ad][0] for ad in MODELLER}
print("\nKurulacak modeller:", ", ".join(KISA_AD[m] for m in MODELLER))

In [ ]:
# =====================================================================
# BOLUM 2 - Metrik hesaplama yardimcilari
# =====================================================================

def guvenli_yuzde_hata(y_true, y_pred, eps=EPS):
    """Sifira bolunmeye karsi guvenli yuzde hata (%). Paydasi eps altinda ise NaN."""
    y_true = np.asarray(y_true, dtype=float)
    payda = np.where(np.abs(y_true) < eps, np.nan, y_true)
    return 100.0 * (np.asarray(y_pred, dtype=float) - y_true) / payda


def metrikleri_hesapla(y_true, y_pred):
    """R2, RMSE, MAE ve (yalnizca ek bilgi olarak) MAPE dondurur."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mape = np.nanmean(np.abs(guvenli_yuzde_hata(y_true, y_pred)))
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": mean_absolute_error(y_true, y_pred),
        "MAPE_%_info_only": float(mape),
    }


def alt_grup_metrikleri(y_true, y_pred, maske):
    """Belirtilen maske icin kayit sayisi, RMSE ve MAE dondurur (R2 hesaplanmaz)."""
    if maske.sum() == 0:
        return {"N_records": 0, "RMSE": np.nan, "MAE": np.nan}
    yt = np.asarray(y_true, dtype=float)[maske]
    yp = np.asarray(y_pred, dtype=float)[maske]
    return {"N_records": int(maske.sum()),
            "RMSE": float(np.sqrt(mean_squared_error(yt, yp))),
            "MAE": float(mean_absolute_error(yt, yp))}

In [ ]:
# =====================================================================
# BOLUM 3 - Onceden tanimlanmis GroupKFold katlariyla capraz dogrulama
# Not: cross_val_predict KULLANILMAZ; dondurulmus fold atamalari kullanilir.
# =====================================================================

def capraz_dogrulama(model_adi, hedef):
    """Her fold icin modeli yeniden kurar, egitir ve validation metriklerini dondurur."""
    satirlar = []
    y = y_train[hedef]

    for fold in range(1, N_SPLITS + 1):
        val_idx = np.where(fold_train.values == fold)[0]
        tr_idx = np.where(fold_train.values != fold)[0]

        # Fold bagimsizlik kontrolu (her fold'da yeniden dogrulanir)
        if set(geo_train.iloc[tr_idx]) & set(geo_train.iloc[val_idx]):
            raise ValueError(f"VERI SIZINTISI: Fold {fold} icinde ortak geometri var.")

        pipe = clone(pipeline_olustur(model_adi))   # scaler yalnizca egitim fold'unda fit edilir
        pipe.fit(X_train.iloc[tr_idx], y.iloc[tr_idx])
        tahmin = pipe.predict(X_train.iloc[val_idx])

        satir = {"Target": TARGET_KISA[hedef], "Model": model_adi, "Fold": fold,
                 "Train_samples": len(tr_idx), "Validation_samples": len(val_idx)}
        satir.update(metrikleri_hesapla(y.iloc[val_idx], tahmin))
        satirlar.append(satir)

    return pd.DataFrame(satirlar)


def cv_ozetle(fold_tablosu):
    """Fold sonuclarindan ortalama ve standart sapma ozeti uretir."""
    ozet = (fold_tablosu.groupby(["Target", "Model"])
            .agg(CV_R2_mean=("R2", "mean"), CV_R2_std=("R2", "std"),
                 CV_RMSE_mean=("RMSE", "mean"), CV_RMSE_std=("RMSE", "std"),
                 CV_MAE_mean=("MAE", "mean"), CV_MAE_std=("MAE", "std"),
                 CV_MAPE_mean_info_only=("MAPE_%_info_only", "mean"))
            .reset_index())
    return ozet


cv_fold_sonuclari, cv_ozetleri = [], []
print("\n--- Capraz dogrulama basliyor ---")
for hedef in TARGETS:
    for model_adi in MODELLER:
        tablo = capraz_dogrulama(model_adi, hedef)
        cv_fold_sonuclari.append(tablo)
        print(f"  {TARGET_KISA[hedef]} | {KISA_AD[model_adi]:6s} | 5 fold tamamlandi")

cv_fold_df = pd.concat(cv_fold_sonuclari, ignore_index=True)
cv_ozet_df = cv_ozetle(cv_fold_df).round(6)

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Baseline_CV_Results.xlsx"),
                    engine="openpyxl") as writer:
    for kisa in ["f1", "f2"]:
        cv_ozet_df[cv_ozet_df["Target"] == kisa].to_excel(
            writer, sheet_name=f"{kisa}_CV_summary", index=False)
        cv_fold_df[cv_fold_df["Target"] == kisa].round(6).to_excel(
            writer, sheet_name=f"{kisa}_CV_folds", index=False)
print("Baseline_CV_Results.xlsx olusturuldu.")

In [ ]:
# =====================================================================
# BOLUM 4 - Tum egitim kumesinde egitim + bagimsiz test degerlendirmesi
# =====================================================================

egitilmis = {}          # (hedef, model) -> egitilmis pipeline
tahminler = {}          # (hedef, model) -> test tahminleri
tt_satirlari, altgrup_satirlari = [], []

print("\n--- Egitim ve test degerlendirmesi ---")
for hedef in TARGETS:
    for model_adi in MODELLER:
        pipe = clone(pipeline_olustur(model_adi))
        pipe.fit(X_train, y_train[hedef])
        egitilmis[(hedef, model_adi)] = pipe

        tahmin_tr = pipe.predict(X_train)
        tahmin_te = pipe.predict(X_test)
        tahminler[(hedef, model_adi)] = tahmin_te

        m_tr = metrikleri_hesapla(y_train[hedef], tahmin_tr)
        m_te = metrikleri_hesapla(y_test[hedef], tahmin_te)

        tt_satirlari.append({
            "Target": TARGET_KISA[hedef], "Model": model_adi,
            "Train_R2": m_tr["R2"], "Train_RMSE": m_tr["RMSE"], "Train_MAE": m_tr["MAE"],
            "Test_R2": m_te["R2"], "Test_RMSE": m_te["RMSE"], "Test_MAE": m_te["MAE"],
            "R2_gap_train_minus_test": m_tr["R2"] - m_te["R2"],
            "RMSE_ratio_test_over_train": (m_te["RMSE"] / m_tr["RMSE"]
                                           if m_tr["RMSE"] > EPS else np.nan),
            "Test_MAPE_%_info_only": m_te["MAPE_%_info_only"],
        })

        # Tasarim uzayi ici / disi alt grup metrikleri
        for etiket, maske in [("Inside training range", ~disarida_mask),
                              ("Outside training range", disarida_mask)]:
            satir = {"Target": TARGET_KISA[hedef], "Model": model_adi, "Subgroup": etiket}
            satir.update(alt_grup_metrikleri(y_test[hedef], tahmin_te, maske))
            altgrup_satirlari.append(satir)

        print(f"  {TARGET_KISA[hedef]} | {KISA_AD[model_adi]:6s} | tamamlandi")

train_test_df = pd.DataFrame(tt_satirlari).round(6)
altgrup_df = pd.DataFrame(altgrup_satirlari).round(6)

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Baseline_Train_Test_Results.xlsx"),
                    engine="openpyxl") as writer:
    for kisa in ["f1", "f2"]:
        train_test_df[train_test_df["Target"] == kisa].to_excel(
            writer, sheet_name=f"{kisa}_train_test", index=False)
    egitim_araligi.round(6).to_excel(writer, sheet_name="Training_feature_ranges", index=False)

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Baseline_Range_Subgroup_Results.xlsx"),
                    engine="openpyxl") as writer:
    for kisa in ["f1", "f2"]:
        altgrup_df[altgrup_df["Target"] == kisa].to_excel(
            writer, sheet_name=f"{kisa}_subgroups", index=False)
    pd.DataFrame({"Outside_range_geometries": sorted(bulunan)}).to_excel(
        writer, sheet_name="Outside_geometries", index=False)

# ---- Test tahmin dosyasi (her model icin ayri sayfa) ------------------
def tahmin_tablosu(model_adi):
    """Bir model icin f1 ve f2 test tahminlerini tek tabloda birlestirir."""
    p1 = tahminler[("f1 (Hz)", model_adi)]
    p2 = tahminler[("f2 (Hz)", model_adi)]
    a1 = y_test["f1 (Hz)"].values
    a2 = y_test["f2 (Hz)"].values
    return pd.DataFrame({
        "Geometry_ID": geo_test.values,
        "Actual_f1_Hz": a1, "Predicted_f1_Hz": p1,
        "Absolute_error_f1_Hz": np.abs(p1 - a1),
        "Percentage_error_f1_%": guvenli_yuzde_hata(a1, p1),
        "Actual_f2_Hz": a2, "Predicted_f2_Hz": p2,
        "Absolute_error_f2_Hz": np.abs(p2 - a2),
        "Percentage_error_f2_%": guvenli_yuzde_hata(a2, p2),
        "Outside_Training_Range": outside_flag,
        "Model": model_adi,
    }).round(6)


with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Baseline_Test_Predictions.xlsx"),
                    engine="openpyxl") as writer:
    for model_adi in MODELLER:
        tahmin_tablosu(model_adi).to_excel(
            writer, sheet_name=f"{KISA_AD[model_adi]}_predictions", index=False)
print("Tahmin ve sonuc dosyalari olusturuldu.")

In [ ]:
# =====================================================================
# BOLUM 5 - Model siralamasi (test RMSE -> test R2 -> test MAE)
# Not: Nihai model secimi YAPILMAZ; yalnizca aday siralamasi uretilir.
# =====================================================================

def siralama_tablosu(hedef_kisa):
    """CV ve test sonuclarini birlestirip belirtilen oncelik sirasina gore siralar."""
    cv = cv_ozet_df[cv_ozet_df["Target"] == hedef_kisa]
    tt = train_test_df[train_test_df["Target"] == hedef_kisa]
    birlesik = cv.merge(tt, on=["Target", "Model"], how="inner")

    birlesik = birlesik.sort_values(
        by=["Test_RMSE", "Test_R2", "Test_MAE"],
        ascending=[True, False, True]).reset_index(drop=True)
    birlesik.insert(0, "Rank", np.arange(1, len(birlesik) + 1))
    birlesik["Optimization_candidate"] = np.where(birlesik["Rank"] <= 3, "Yes", "No")
    return birlesik.round(6)


siralamalar = {k: siralama_tablosu(k) for k in ["f1", "f2"]}

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Baseline_Model_Ranking.xlsx"),
                    engine="openpyxl") as writer:
    for kisa, tablo in siralamalar.items():
        tablo.to_excel(writer, sheet_name=f"{kisa}_ranking", index=False)
    pd.DataFrame([
        {"Note": "Ranking priority: lowest Test RMSE, then highest Test R2, then lowest Test MAE."},
        {"Note": "No final model selection is made at this stage."},
        {"Note": "MAPE is reported for information only and is not used for ranking."},
    ]).to_excel(writer, sheet_name="Notes", index=False)

print("\n--- Test RMSE'ye gore ilk uc model ---")
for kisa, tablo in siralamalar.items():
    print(f"  {kisa}: " + ", ".join(KISA_AD[m] for m in tablo.loc[:2, "Model"]))

In [ ]:
# =====================================================================
# BOLUM 6 - Makale kalitesinde grafikler
# =====================================================================

METRIK_ETIKET = {"R2": "$R^2$ (-)", "RMSE": "RMSE (Hz)", "MAE": "MAE (Hz)"}
HEDEF_ETIKET = {"f1": "$f_1$", "f2": "$f_2$"}


def model_karsilastirma_grafigi(hedef_kisa, metrik, kaynak):
    """Modelleri belirtilen metrige gore karsilastirir. kaynak: 'CV' veya 'Test'."""
    if kaynak == "CV":
        veri = cv_ozet_df[cv_ozet_df["Target"] == hedef_kisa]
        deger = veri[f"CV_{metrik}_mean"].values
        hata = veri[f"CV_{metrik}_std"].values
        renk, baslik_kaynak = RENK_CV, "Cross-validation"
    else:
        veri = train_test_df[train_test_df["Target"] == hedef_kisa]
        deger = veri[f"Test_{metrik}"].values
        hata = None
        renk, baslik_kaynak = RENK_TEST, "Independent test set"

    etiketler = [KISA_AD[m] for m in veri["Model"]]
    fig, ax = plt.subplots(figsize=(7.0, 4.4))
    ax.bar(etiketler, deger, yerr=hata, capsize=4, color=renk,
           edgecolor="white", width=0.62,
           error_kw=dict(ecolor="#333333", lw=1.0))
    ax.set_ylabel(f"{baslik_kaynak} {METRIK_ETIKET[metrik]}")
    ax.set_xlabel("Regression model")
    ax.set_title(f"Target: {HEDEF_ETIKET[hedef_kisa]}", loc="left")
    if metrik == "R2":
        ax.axhline(0, color="#333333", lw=0.9)
    sns.despine(ax=ax)
    fig.tight_layout()
    kaydet(fig, COMP_DIR, f"Fig_{kaynak}_{metrik}_comparison_{hedef_kisa}")


def gercek_tahmin_grafigi(hedef, model_adi):
    """y = x referans dogrusu ve metriklerle actual-vs-predicted grafigi."""
    kisa = TARGET_KISA[hedef]
    gercek = y_test[hedef].values
    tahmin = tahminler[(hedef, model_adi)]
    m = metrikleri_hesapla(gercek, tahmin)

    fig, ax = plt.subplots(figsize=(5.4, 5.2))
    ic = ~disarida_mask
    ax.scatter(gercek[ic], tahmin[ic], s=26, color=RENK_CV, alpha=0.6,
               edgecolors="none", label="Inside training range")
    ax.scatter(gercek[disarida_mask], tahmin[disarida_mask], s=52, marker="^",
               facecolors="none", edgecolors=RENK_VURGU, linewidths=1.4,
               label="Outside training range")

    alt = min(gercek.min(), tahmin.min())
    ust = max(gercek.max(), tahmin.max())
    pay = 0.05 * (ust - alt)
    ax.plot([alt - pay, ust + pay], [alt - pay, ust + pay],
            color="#333333", lw=1.2, ls="--", label="y = x")
    ax.set_xlim(alt - pay, ust + pay)
    ax.set_ylim(alt - pay, ust + pay)
    ax.set_aspect("equal", adjustable="box")

    ax.text(0.04, 0.96,
            f"$R^2$ = {m['R2']:.3f}\nRMSE = {m['RMSE']:.4f} Hz\nMAE = {m['MAE']:.4f} Hz",
            transform=ax.transAxes, va="top", ha="left", fontsize=9.5,
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white",
                      edgecolor="0.8", alpha=0.9))
    ax.set_xlabel(f"Actual {HEDEF_ETIKET[kisa]} (Hz)")
    ax.set_ylabel(f"Predicted {HEDEF_ETIKET[kisa]} (Hz)")
    ax.set_title(model_adi, loc="left")
    ax.legend(frameon=False, loc="lower right")
    sns.despine(ax=ax)
    fig.tight_layout()
    kaydet(fig, AVP_DIR, f"Fig_actual_vs_predicted_{kisa}_{KISA_AD[model_adi]}")


def artik_grafigi(hedef, model_adi):
    """Predicted vs residual grafigi (residual = actual - predicted)."""
    kisa = TARGET_KISA[hedef]
    gercek = y_test[hedef].values
    tahmin = tahminler[(hedef, model_adi)]
    artik = gercek - tahmin

    fig, ax = plt.subplots(figsize=(5.8, 4.6))
    ic = ~disarida_mask
    ax.scatter(tahmin[ic], artik[ic], s=26, color=RENK_CV, alpha=0.6,
               edgecolors="none", label="Inside training range")
    ax.scatter(tahmin[disarida_mask], artik[disarida_mask], s=52, marker="^",
               facecolors="none", edgecolors=RENK_VURGU, linewidths=1.4,
               label="Outside training range")
    ax.axhline(0, color="#333333", lw=1.1, ls="--")
    ax.set_xlabel(f"Predicted {HEDEF_ETIKET[kisa]} (Hz)")
    ax.set_ylabel(f"Residual (actual - predicted) {HEDEF_ETIKET[kisa]} (Hz)")
    ax.set_title(model_adi, loc="left")
    ax.legend(frameon=False, loc="best")
    sns.despine(ax=ax)
    fig.tight_layout()
    kaydet(fig, RES_DIR, f"Fig_residuals_{kisa}_{KISA_AD[model_adi]}")


# ---- Grafiklerin uretilmesi -------------------------------------------
print("\n--- Grafikler uretiliyor ---")
for kisa in ["f1", "f2"]:
    for metrik in ["R2", "RMSE", "MAE"]:
        for kaynak in ["CV", "Test"]:
            model_karsilastirma_grafigi(kisa, metrik, kaynak)

# Test RMSE'ye gore her hedef icin ayri ilk uc model
for hedef in TARGETS:
    kisa = TARGET_KISA[hedef]
    ilk_uc = siralamalar[kisa].loc[:2, "Model"].tolist()
    print(f"  {kisa} icin ilk uc model: {[KISA_AD[m] for m in ilk_uc]}")
    for model_adi in ilk_uc:
        gercek_tahmin_grafigi(hedef, model_adi)
        artik_grafigi(hedef, model_adi)

print("\nAsama 4 tamamlandi. Hicbir optimizasyon yapilmadi, nihai model secilmedi.")
print("Cikti klasoru:", OUTPUT_DIR)